# 🇲🇬 Serveur de Traduction EN/FR vers Malgache (Kaggle)

Ce notebook lance le backend FastAPI sur **Kaggle avec GPU gratuit (T4 ou P100)** et expose l'API publiquement via **ngrok**.

### ⚙️ Configuration préalable sur Kaggle (Essentiel) :
1. **Activer Internet** : Dans le panneau latéral droit (*Session options*), cochez **Internet ON**.
2. **Activer le GPU** : Dans *Session options* > *Accelerator*, sélectionnez **GPU T4 x2** ou **GPU P100**.
3. **Ajouter les 2 Secrets ngrok et API_KEY** :
   - Dans le menu supérieur de Kaggle : cliquez sur **Add-ons** > **Secrets**.
   - Secret 1 : Label `NGROK_AUTHTOKEN`, Value: votre token depuis [dashboard.ngrok.com](https://dashboard.ngrok.com).
   - Secret 2 : Label `API_KEY`, Value: votre mot de passe/clé secrète pour protéger l'API.
   - Cochez les cases pour lier les secrets à ce notebook, puis cliquez sur **Save**.

In [ ]:
# Récupération du dépôt GitHub dans l'espace de travail Kaggle
REPO_URL = "https://github.com/franck504/en-fr-to-mg-api.git"
BRANCH = "main"
PROJECT_DIR = "/kaggle/working/en_fr_to_MG"

import os
import subprocess

if os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print(f"Projet prêt dans {PROJECT_DIR}")

In [ ]:
# Installation des dépendances (PyTorch et CUDA sont déjà pré-installés dans Kaggle)
!pip install -q -U -r requirements-colab.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# 🔒 Récupération du token ngrok depuis Kaggle Secrets
try:
    os.environ["NGROK_AUTHTOKEN"] = user_secrets.get_secret("NGROK_AUTHTOKEN")
    print("✅ Secret NGROK_AUTHTOKEN chargé avec succès !")
except Exception as exc:
    raise RuntimeError(
        "Veuillez ajouter le secret 'NGROK_AUTHTOKEN' via le menu Add-ons > Secrets. "
        "Obtenez votre token gratuit sur https://dashboard.ngrok.com"
    ) from exc

# 🔒 Récupération de la clé API depuis Kaggle Secrets
try:
    os.environ["API_KEY"] = user_secrets.get_secret("API_KEY")
    print("✅ Secret API_KEY chargé avec succès !")
except Exception as exc:
    raise RuntimeError(
        "Veuillez ajouter le secret 'API_KEY' via le menu Add-ons > Secrets avec la clé de votre choix."
    ) from exc

# Configuration du serveur pour l'environnement Kaggle avec pré-chargement GPU
os.environ["ENABLE_NGROK"] = "true"
os.environ["HOST"] = "0.0.0.0"
os.environ["PORT"] = "8000"
os.environ["PROVIDER"] = "hf_seq2seq"
os.environ["HF_MODEL_NAME"] = "facebook/nllb-200-distilled-600M"
os.environ["HF_MODEL_FAMILY"] = "nllb"
os.environ["HF_DEVICE"] = "auto"
os.environ["MODEL_CACHE_DIR"] = "/kaggle/working/hf_models"
os.environ["LOAD_MODEL_ON_STARTUP"] = "true"

print("🚀 Configuration validée pour Kaggle.")

In [ ]:
# Démarrage du serveur, pré-chargement GPU et ouverture du tunnel ngrok
!python run_colab.py

## 💡 Utilisation de votre API sécurisée

Remplacez `VOTRE_CLE_API` par la valeur que vous avez configurée dans le secret `API_KEY`.

### Exemple Python avec SDK OpenAI :
```python
from openai import OpenAI

client = OpenAI(
    base_url="https://votre-id.ngrok-free.app/v1",
    api_key="VOTRE_CLE_API",
)
response = client.chat.completions.create(
    model="nllb",
    messages=[{"role": "user", "content": "Good morning my friend"}],
)
print(response.choices[0].message.content)
```

### Exemple cURL direct :
```bash
curl -X POST https://votre-id.ngrok-free.app/translate \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer VOTRE_CLE_API" \
  -d '{"text": "Bonjour, comment allez-vous ?", "source_lang": "auto", "target_lang": "mg"}'
```